# Clase 15 · Notebook 02
# Aumentación de datos y batch normalization

**Introducción al Deep Learning — Módulo 3, Unidad 1: Redes convolucionales (Parte II)**

Dos técnicas que ayudan a que una CNN generalice mejor:

1. **Aumentación de datos**: crear imágenes nuevas transformando las que tenemos.
2. **Batch normalization**: normalizar las activaciones para un entrenamiento estable.

Comprobamos que la aumentación **mejora** cuando hay pocos datos de entrenamiento.

> 💡 En Colab, TensorFlow ya viene instalado. Ejecuta las celdas en orden con **Shift + Enter**.

## 1. Datos (pocos ejemplos, para que la aumentación se note)

In [ ]:
try:
    import tensorflow as tf
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
    import tensorflow as tf
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
tf.random.set_seed(42); np.random.seed(42)

digits = load_digits()
X = digits.images.reshape(-1, 8, 8, 1).astype("float32") / 16.0
y = tf.keras.utils.to_categorical(digits.target, 10)
# Entrenamiento PEQUEÑO para que la aumentación aporte
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.85, random_state=42, stratify=digits.target)
print("Entrenamiento:", Xtr.shape[0], "| Test:", Xte.shape[0])

## 2. Ver la aumentación en acción

Las capas de aumentación de Keras aplican pequeñas transformaciones aleatorias (rotación, zoom,
desplazamiento). Cada vez que pasa una imagen, sale ligeramente distinta.

In [ ]:
from tensorflow.keras import layers

aumentacion = tf.keras.Sequential([
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1),
])

img = Xtr[0:1]
fig, axes = plt.subplots(1, 6, figsize=(11, 2))
axes[0].imshow(img[0].reshape(8, 8), cmap="gray"); axes[0].set_title("original"); axes[0].axis("off")
for ax in axes[1:]:
    ax.imshow(aumentacion(img, training=True)[0].numpy().reshape(8, 8), cmap="gray")
    ax.set_title("aumentada"); ax.axis("off")
plt.tight_layout(); plt.show()

## 3. Entrenar dos CNN: sin y con aumentación (+ batch norm)

La segunda red incluye la capa de aumentación al principio y **BatchNormalization** entre las convoluciones.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Input, BatchNormalization

def cnn(con_aumentacion=False):
    tf.random.set_seed(42)
    capas = [Input(shape=(8, 8, 1))]
    if con_aumentacion:
        capas += [layers.RandomRotation(0.1), layers.RandomZoom(0.1), layers.RandomTranslation(0.1, 0.1)]
    capas += [
        Conv2D(16, (3, 3), padding="same", activation="relu"), BatchNormalization(),
        MaxPooling2D((2, 2)),
        Conv2D(32, (3, 3), padding="same", activation="relu"), BatchNormalization(),
        MaxPooling2D((2, 2)),
        Flatten(), Dense(32, activation="relu"), Dense(10, activation="softmax"),
    ]
    m = Sequential(capas)
    m.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return m

m_sin = cnn(False); m_sin.fit(Xtr, ytr, epochs=25, batch_size=32, verbose=0)
m_con = cnn(True);  m_con.fit(Xtr, ytr, epochs=25, batch_size=32, verbose=0)

print("Accuracy en test (pocos datos):")
print("  Sin aumentación : %.1f %%" % (m_sin.evaluate(Xte, yte, verbose=0)[1] * 100))
print("  Con aumentación : %.1f %%" % (m_con.evaluate(Xte, yte, verbose=0)[1] * 100))

Con un conjunto de entrenamiento pequeño, la **aumentación** suele mejorar la accuracy en test: la red
ve más variedad y generaliza mejor. Nota: en inferencia (test) la aumentación se **desactiva** automáticamente.

## Experimenta tú

1. Sube la intensidad de la aumentación (p. ej. `RandomRotation(0.2)`).
2. Quita la `BatchNormalization` y compara.
3. Usa más datos de entrenamiento: ¿sigue ayudando la aumentación tanto?

## Resumen

- La **aumentación de datos** genera variantes de las imágenes (rotación, zoom, desplazamiento) y amplía el
  conjunto de entrenamiento, mejorando la generalización (sobre todo con pocos datos).
- Se aplica **solo en entrenamiento**; en inferencia se desactiva.
- La **batch normalization** estabiliza y acelera el entrenamiento.

Con esto cerramos la unidad de redes convolucionales.